In [48]:
# !rm ./data/playground-series-s6e1.zip
# !rm -rf ./data/playground-series-s6e1
# !kaggle competitions download -c playground-series-s6e1 -p ./data
# !unzip ./data/playground-series-s6e1.zip -d ./data/playground-series-s6e1
# !rm ./data/playground-series-s6e1.zip

In [49]:
import pandas as pd

# split into train and test
from sklearn.model_selection import train_test_split

# standardize study_hours, class_attendance, sleep_hours, exam_score to have mean 0 and std 1
from sklearn.preprocessing import StandardScaler


In [50]:
df = pd.read_csv("./data/playground-series-s6e1/train.csv")
df.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0


In [51]:
# find missing values
df.isnull().sum()

id                  0
age                 0
gender              0
course              0
study_hours         0
class_attendance    0
internet_access     0
sleep_hours         0
sleep_quality       0
study_method        0
facility_rating     0
exam_difficulty     0
exam_score          0
dtype: int64

In [52]:
NUMERIC_COLS = ["study_hours", "class_attendance", "sleep_hours"]


def preprocess(df, scaler=None):
    """
    Encode and scale features.
    scaler=None  → fit a new scaler (training)
    scaler=...   → apply existing scaler (inference/submission)
    Returns (processed_df, scaler)
    """
    df = df.copy()
    if "id" in df.columns:
        df = df.drop("id", axis=1)

    df["age"] = df["age"].astype(int)
    df["is_female"] = df["gender"].apply(lambda x: 1 if x == "female" else 0)
    df = df.drop("gender", axis=1)
    df = pd.get_dummies(df, columns=["course"], prefix="course", dtype=int)
    df["has_internet"] = df["internet_access"].apply(lambda x: 1 if x == "yes" else 0)
    df = df.drop("internet_access", axis=1)
    df["sleep_quality"] = df["sleep_quality"].map({"poor": 0, "average": 1, "good": 2})
    df = pd.get_dummies(df, columns=["study_method"], prefix="study_method", dtype=int)
    df["facility_rating"] = df["facility_rating"].map(
        {"low": 0, "medium": 1, "high": 2}
    )
    df["exam_difficulty"] = df["exam_difficulty"].map(
        {"easy": 0, "moderate": 1, "hard": 2}
    )

    if scaler is None:
        scaler = StandardScaler()
        df[NUMERIC_COLS] = scaler.fit_transform(df[NUMERIC_COLS])
    else:
        df[NUMERIC_COLS] = scaler.transform(df[NUMERIC_COLS])

    return df, scaler

In [53]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_df, scaler = preprocess(train_df)
test_df, _ = preprocess(test_df, scaler=scaler)

y_train = train_df["exam_score"]
X_train = train_df.drop("exam_score", axis=1)
y_test = test_df["exam_score"]
X_test = test_df.drop("exam_score", axis=1)


In [54]:
# build a neural network with PyTorch to predict exam_score from the other columns
import torch
import torch.nn as nn
import torch.optim as optim


class StudentScorePredictor(nn.Module):
    def __init__(self, input_size):
        super(StudentScorePredictor, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


input_size = X_train.shape[1]
model = StudentScorePredictor(input_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# convert X_train and y_train to torch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# train the model for 500 epochs
num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

# evaluate the model on the test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)
    print(f"Test Loss: {test_loss.item():.4f}")


Epoch [10/500], Loss: 4068.2803
Epoch [20/500], Loss: 3821.8691
Epoch [30/500], Loss: 3488.5725
Epoch [40/500], Loss: 3039.4514
Epoch [50/500], Loss: 2464.6824
Epoch [60/500], Loss: 1793.8981
Epoch [70/500], Loss: 1120.6232
Epoch [80/500], Loss: 596.7941
Epoch [90/500], Loss: 347.6670
Epoch [100/500], Loss: 313.6145
Epoch [110/500], Loss: 312.1283
Epoch [120/500], Loss: 295.1660
Epoch [130/500], Loss: 284.5362
Epoch [140/500], Loss: 277.6691
Epoch [150/500], Loss: 270.6150
Epoch [160/500], Loss: 263.9249
Epoch [170/500], Loss: 257.6811
Epoch [180/500], Loss: 251.6447
Epoch [190/500], Loss: 245.7997
Epoch [200/500], Loss: 240.1319
Epoch [210/500], Loss: 234.6129
Epoch [220/500], Loss: 229.2290
Epoch [230/500], Loss: 223.9692
Epoch [240/500], Loss: 218.8245
Epoch [250/500], Loss: 213.7883
Epoch [260/500], Loss: 208.8553
Epoch [270/500], Loss: 204.0217
Epoch [280/500], Loss: 199.2847
Epoch [290/500], Loss: 194.6407
Epoch [300/500], Loss: 190.0866
Epoch [310/500], Loss: 185.6194
Epoch [320

In [55]:
# build a sample submission file from the example submission file ./data/playground-series-s6e1/sample_submission.csv
sample_submission = pd.read_csv("./data/playground-series-s6e1/sample_submission.csv")
sample_submission.head()

,id,exam_score
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [56]:
submission_df = pd.read_csv("./data/playground-series-s6e1/test.csv")
ids = submission_df["id"]

submission_features, _ = preprocess(submission_df, scaler=scaler)

model.eval()
with torch.no_grad():
    sub_tensor = torch.tensor(submission_features.values, dtype=torch.float32)
    sub_preds = model(sub_tensor).numpy().flatten()

pd.DataFrame({"id": ids, "exam_score": sub_preds}).to_csv(
    "./data/playground-series-s6e1/submission.csv", index=False
)


In [ ]:
# !kaggle competitions submit -c playground-series-s6e1 -f ./data/playground-series-s6e1/submission.csv -m "Message"

100%|███████████████████████████████████████| 4.22M/4.22M [00:04<00:00, 924kB/s]
Successfully submitted to Predicting Student Test Scores